# xgb + lgb

In [1]:
import pandas as pd

In [2]:
from dataengineers import Dataset

In [3]:
dataset = Dataset('train')
train, test = dataset.build_train_test()

In [4]:
exclude = ['id', 'target', 'delivery_start']

In [5]:
features = [c for c in train.columns if c not in exclude]

In [6]:
ds2 = Dataset('test')
df_out = ds2.build_main()

## lgbm

In [7]:
from models import lGBM

In [8]:
lg = lGBM(features)

In [9]:
lg.fit(train, test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002341 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18563
[LightGBM] [Info] Number of data points in the train set: 106086, number of used features: 83
[LightGBM] [Info] Start training from score 36.843101
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Lig

In [10]:
lg_vals = lg.predict(df_out)

## XGB

In [11]:
from models import XGB

In [12]:
xg = XGB(features)

In [13]:
xg.fit(train, test)

[0]	validation_0-rmse:152.09925	validation_1-rmse:62.87876
[100]	validation_0-rmse:119.27332	validation_1-rmse:53.46553
[200]	validation_0-rmse:100.68952	validation_1-rmse:49.32961
[300]	validation_0-rmse:88.70899	validation_1-rmse:47.05757
[400]	validation_0-rmse:79.90878	validation_1-rmse:45.53266
[500]	validation_0-rmse:73.31926	validation_1-rmse:44.48206
[600]	validation_0-rmse:67.99144	validation_1-rmse:43.80065
[700]	validation_0-rmse:63.54227	validation_1-rmse:43.25580
[800]	validation_0-rmse:59.95068	validation_1-rmse:43.01127
[900]	validation_0-rmse:56.88678	validation_1-rmse:42.79455
[1000]	validation_0-rmse:54.24962	validation_1-rmse:42.71253
[1047]	validation_0-rmse:53.14750	validation_1-rmse:42.75482


In [14]:
xg_vals = xg.predict(df_out)

/home/matt/repos/nitor-comp/.venv/lib/python3.14/site-packages/xgboost/core.py:751: UserWarning: [16:30:00] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


## submission

In [15]:
from utils import Ensemble

In [16]:
ensem = Ensemble([0.7, 0.3], xg_vals, lg_vals)

In [17]:
y_out = ensem.build()

In [18]:
len(y_out)

13098

In [19]:
df_out['target'] = y_out

In [20]:
df_out.head()

,id,global_horizontal_irradiance,diffuse_horizontal_irradiance,direct_normal_irradiance,cloud_cover_total,cloud_cover_low,cloud_cover_mid,cloud_cover_high,precipitation_amount,visibility,...,load_forecast_lag_12h,load_forecast_lag_24h,load_forecast_lag_48h,load_forecast_lag_168h,load_forecast_rolling_mean_6h,load_forecast_rolling_mean_24h,load_forecast_ramp_1h,load_forecast_ramp_3h,load_forecast_ramp_6h,target
0,133627,0.0,0.0,0.0,100.0,14.0,44.0,100.0,0.0,16600.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,113.726065
1,133635,0.0,0.0,0.0,100.0,100.0,100.0,88.0,0.0,13800.0,...,NaN,NaN,NaN,NaN,55189.474200,55189.474200,-2203.0271,NaN,NaN,71.734973
2,133643,0.0,0.0,0.0,100.0,70.0,100.0,100.0,0.0,19700.0,...,NaN,NaN,NaN,NaN,54087.960650,54087.960650,-1707.9816,NaN,NaN,126.565046
3,133651,0.0,0.0,0.0,99.0,13.0,99.0,94.0,0.0,16200.0,...,NaN,NaN,NaN,NaN,53151.462267,53151.462267,-893.3085,-4804.3172,NaN,137.144288
4,133659,0.0,0.0,0.0,96.0,0.0,96.0,79.0,0.0,14500.0,...,NaN,NaN,NaN,NaN,52459.885950,52459.885950,-439.4928,-3040.7829,NaN,133.397930


In [21]:
from utils import Submission

In [22]:
submit = Submission(df_out)

In [23]:
submit.process()

,id,target
0,133627,113.726065
2183,133629,60.092767
4366,133630,60.683406
10915,133631,54.155370
6549,133633,50.513432
...,...,...
4365,146774,61.707746
6548,146775,32.622912
13097,146776,34.278133
8731,146777,38.249607


In [24]:
submit.validate()

✅ Validation passed!


In [25]:
submit.dump()